# Information and identifiability: the un-inverted view

`information(model)` is the sibling of `covariance(model)`: the reduced
Hessian over the fitted block, un-inverted, natural units, from the same
single solve. Two situations where it says more than the covariance:

1. A POORLY IDENTIFIED fit. When the data barely distinguishes two parameters, the
   covariance matrix shows everything correlated with everything; the
   information matrix's `eigen()` finds the near-zero eigenvalue and its
   eigenvector NAMES the combination the data cannot pin down
   (here: trade `a` for `b`, compensated by moving `k2`).
2. A parameter at a bound. The covariance reports zero variance
   (conditional on the bound), which hides how much the data actually
   says; the information matrix returns S, the reduction onto the
   pinned set, because zero variance is NOT zero information.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
import pyomo.environ as pyo

import pyomo_pounce  # registers 'pounce' with SolverFactory
from pyomo_pounce import (
    covariance, declare_fitted, declare_residual, information)

rng = np.random.default_rng(11)
A_TRUE, B_TRUE, K1_TRUE, K2_TRUE = 2.0, 1.0, 0.8, 1.6
t_data = np.linspace(0.0, 5.0, 40)
y_data = (A_TRUE * np.exp(-K1_TRUE * t_data)
          + B_TRUE * np.exp(-K2_TRUE * t_data)
          + 0.02 * rng.standard_normal(len(t_data)))

## 1. A poorly identified two-exponential fit

Two decaying exponentials with rates within a factor of two of each
other: a textbook poorly identified model. Each parameter's marginal uncertainty
looks alarming and the correlation matrix is nearly singular, but
neither view says WHICH direction is the problem.

In [2]:
def build(t, y, declare=True):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(0, len(t) - 1)
    m.a = pyo.Var(initialize=1.8)
    m.b = pyo.Var(initialize=1.2)
    m.k1 = pyo.Var(initialize=0.7)
    m.k2 = pyo.Var(initialize=1.8)
    m.r = pyo.Var(m.I, initialize=0.0)
    m.res = pyo.Constraint(
        m.I, rule=lambda mm, i: mm.r[i]
        == float(y[i]) - mm.a * pyo.exp(-mm.k1 * float(t[i]))
        - mm.b * pyo.exp(-mm.k2 * float(t[i])))
    m.obj = pyo.Objective(expr=sum(m.r[i] ** 2 for i in m.I))
    if declare:
        declare_fitted(m.a, m.b, m.k1, m.k2)
        declare_residual(m.r)
    return m

m = build(t_data, y_data)
pyo.SolverFactory("pounce").solve(m, tee=True)
cov = covariance(m)
for v in (m.a, m.b, m.k1, m.k2):
    print(f"{v.name:3s} = {pyo.value(v):7.4f} +/- {cov.std_err[v]:.4f}")
print("\ncorrelation:")
print(np.array2string(cov.correlation.matrix, precision=3,
                      suppress_small=True))

********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.9.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:      198
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       44

Total number of variables............................:       44
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       40
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 7.41e-02 0.00e+00   -1.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1  1.0560637e-02 1.42e-02 9.86e-04   -2.5 2.85e-01      - 1.0

a   =  2.4046 +/- 0.2910
b   =  0.6099 +/- 0.2864
k1  =  0.8611 +/- 0.0390
k2  =  1.9502 +/- 0.4834

correlation:
[[ 1.    -0.999  0.985  0.974]
 [-0.999  1.    -0.986 -0.967]
 [ 0.985 -0.986  1.     0.929]
 [ 0.974 -0.967  0.929  1.   ]]


## 2. `eigen()` names the poorly identified combination

The information matrix's eigenvalues span several orders of magnitude:
the small ones are the directions the data does not inform. The
eigenvector of the smallest eigenvalue is the parameter combination the
fit cannot distinguish, in so many words: trade `a` against `b` while
shifting the rates toward each other.

In [3]:
info = information(m)
ev, vec = info.eigen()
names = [v.name for v in info.params]
print("information eigenvalues (ascending):")
print(np.array2string(ev, precision=3))
print(f"\nspread: {ev[-1] / ev[0]:.1e}")
print("\nleast identified direction (eigenvector of the smallest eigenvalue):")
for n, c in zip(names, vec[:, 0]):
    print(f"   {c:+.3f} * {n}")

information eigenvalues (ascending):
[1.464e-03 1.003e-01 8.574e+00 4.173e+01]

spread: 2.9e+04

least identified direction (eigenvector of the smallest eigenvalue):
   +0.459 * a
   -0.450 * b
   +0.059 * k1
   +0.764 * k2


## 3. The duality, and Gauss-Newton

For this homoscedastic fit the two accessors are exact inverses of one
another, `cov == 2 sigma^2 inv(info)`, and `hessian="gauss-newton"`
gives the expected information `2 J'J` from the same backsolves. Near a
good fit with small residuals the two forms agree; they separate when
residual curvature matters.

In [4]:
dual = 2.0 * cov.sigma_sq * np.linalg.inv(info.matrix)
resid = np.max(np.abs(cov.matrix / dual - 1.0))
print(f"max relative |cov / (2 sigma^2 inv(info)) - 1| = {resid:.2e}")

info_gn = information(m, hessian="gauss-newton")
print(f"max relative Lagrangian vs Gauss-Newton difference: "
      f"{np.max(np.abs(info.matrix / info_gn.matrix - 1)):.2e}")

max relative |cov / (2 sigma^2 inv(info)) - 1| = 1.03e-04
max relative Lagrangian vs Gauss-Newton difference: 5.60e-04


## 4. At a bound: zero variance is not zero information

First, something the poorly identified model itself teaches. Force `b`
against a bound well above its fitted value: the exchange direction
absorbs most of the shove, the multiplier stays weak, and the
classifier honestly refuses the strong call, reporting the activity as
ambiguous and keeping `b` at finite variance. A pin only bites when
the data actually resists it.

In [5]:
b_hat = pyo.value(m.b)
mp = build(t_data, y_data)
mp.b.setlb(b_hat + 1.5)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pyo.SolverFactory("pounce").solve(mp)
    cov_p = covariance(mp)
for w in caught:
    if "ambiguous" in str(w.message):
        print(str(w.message)[:120], "...")
print(f"\nvariance of b, pin absorbed: {cov_p[mp.b]:.6g}   (finite, kept)")

covariance: fitted parameter b has ambiguous bound activity at the solve's final barrier parameter; re-solve with a tigh ...

variance of b, pin absorbed: 0.532899   (finite, kept)


On a cleanly identified fit the same push bites. Pin `A` above its
fitted value: the bound is strongly active, and `covariance()` reports
a zero row for `A`, the variance conditional on the bound. The
information matrix instead returns S, the Schur reduction onto the
pinned parameter: how much the data says about `A` with the free
parameter profiled out. Large, not zero, and that is the point.

In [6]:
def build_simple(t, y):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(0, len(t) - 1)
    m.A = pyo.Var(initialize=1.0)
    m.k = pyo.Var(initialize=1.0)
    m.r = pyo.Var(m.I, initialize=0.0)
    m.res = pyo.Constraint(
        m.I, rule=lambda mm, i: mm.r[i]
        == float(y[i]) - mm.A * pyo.exp(-mm.k * float(t[i])))
    m.obj = pyo.Objective(expr=sum(m.r[i] ** 2 for i in m.I))
    declare_fitted(m.A, m.k)
    declare_residual(m.r)
    return m

y_simple = (2.0 * np.exp(-0.9 * t_data)
            + 0.02 * rng.standard_normal(len(t_data)))
ms = build_simple(t_data, y_simple)
pyo.SolverFactory("pounce").solve(ms)
A_hat = pyo.value(ms.A)

m2 = build_simple(t_data, y_simple)
m2.A.setlb(A_hat + 0.4)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pyo.SolverFactory("pounce").solve(m2)
    cov2 = covariance(m2)
    info2 = information(m2)
for w in caught:
    if "strongly active" in str(w.message):
        print(str(w.message)[:120], "...")
print(f"\ncovariance  of A at the bound: {cov2[m2.A]:.6g}   (zero row)")
print(f"information of A at the bound: {info2[m2.A]:.6g}   (S, not zero)")
print(f"free-block entry, k:           {info2[m2.k]:.6g}")

covariance: fitted parameter A is held by its bound at the optimum (strongly active); its direction is projected out (ze ...
information: fitted parameter A is held by its bound at the optimum (strongly active); its direction is projected out (z ...

covariance  of A at the bound: 0   (zero row)
information of A at the bound: 4.28891   (S, not zero)
free-block entry, k:           15.6849


`covariance()` for prediction and reporting, `information()` for
diagnosis: identifiability directions via `eigen()`, experiment-design
reasoning (information is additive over data), and honest answers at
bounds.